# Step 0.4 — Crop masks from MIRCA2000 (original source), no 5% presence threshold

**Ablation requested by the user**: `1.0.bis)` changed TWO things at once relative to the
original `0.1)` (MIRCA2000 + 5% threshold) -- (1) removed the 5% presence threshold, AND
(2) switched the presence source from MIRCA2000 to MIRCA-OS 2005. That branch fixed the
NE-Europe/Ukraine wheat-winter gap vs GAEZ, but it's unclear which of the two changes did it.

This notebook isolates change (1) alone: **MIRCA2000 stays the presence source**, only the
5% threshold is removed (a 1deg cell counts as 'presence' if MIRCA2000's own raw calendar
records *any* harvested area in it at all, `> 0`, summed over all native 5-arcmin pixels in
the cell -- no per-pixel-share, no per-cell-share threshold). Sowing/maturity DOY are copied
unchanged from the existing (threshold-based) `mirca2000_1deg_<Crop>_rf.nc` files, exactly
like `1.0.bis)` does -- so calendar handling is identical between the two ablations, only the
presence array differs. Unlike `1.0.bis)`, Wheat-spring_rf and Wheat-winter_rf each get their
own presence array here (season-filtered from MIRCA2000's own calendar), since MIRCA2000,
unlike MIRCA-OS's raster, does split wheat into winter/spring sub-crops.

Output: `mirca2000_no_threshold_1deg_<Crop_rf>.nc`, same variable structure as `1.0.bis)`'s
output, parallel/non-destructive -- nothing in `outputs_input_data_1deg/mirca2000_1deg_*.nc`
is touched.


In [1]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb
CAL_FILE = BASE / "0_Input_data" / "masks_crops" / "MIRCA_2000" / "growing_periods_listed" / "CELL_SPECIFIC_CROPPING_CALENDARS.TXT.gz"
ERA5_1DEG = BASE / "1_SVM" / "0_preprocessing_masks" / "outputs_input_data_1deg"

LAT0, LAT1, LON0, LON1 = 35, 71, -25, 44
NLAT, NLON = LAT1 - LAT0 + 1, LON1 - LON0 + 1
NCELL = NLAT * NLON
lat1 = np.arange(LAT0, LAT1 + 1); lon1 = np.arange(LON0, LON1 + 1)
print(f"1deg grid: {NLAT} x {NLON} = {NCELL} cells")

# crop_label -> (MIRCA2000 crop name, irrigation, wheat season filter or None, old mirca2000_1deg file to copy calendar from)
CROP_MAP = {
    'Maize_rf':        ('Maize',  'rf', None,     'mirca2000_1deg_Maize_rf.nc'),
    'Wheat-spring_rf': ('Wheat',  'rf', 'spring', 'mirca2000_1deg_Wheat-spring_rf.nc'),
    'Wheat-winter_rf': ('Wheat',  'rf', 'winter', 'mirca2000_1deg_Wheat-winter_rf.nc'),
    'Potato_rf':       ('Potato', 'rf', None,     'mirca2000_1deg_Potato_rf.nc'),
}


1deg grid: 37 x 70 = 2590 cells


## 1. Target grid + land mask (same source as 0.1 / 1.0.bis, for consistency)

In [2]:
sample_file = ERA5_1DEG / "ERA5_1deg_tasmax_1981_2015_daily.nc"
ds_grid = xr.open_dataset(sample_file)
land_mask = np.isfinite(ds_grid['tasmax'].isel(time=0).values)
ds_grid.close()
print(f"Land 1deg cells: {int(land_mask.sum())} / {land_mask.size}")


Land 1deg cells: 1332 / 2590


## 2. MIRCA2000 presence at 1deg, no threshold (any recorded area > 0, summed over native pixels)

In [3]:
NROW, NCOL = 2160, 4320
ROWS_, COLS_ = np.arange(NROW), np.arange(NCOL)
lat5 = 90 - (ROWS_ + 0.5) / 12
lon5 = -180 + (COLS_ + 0.5) / 12
lat5_i = np.floor(lat5 + 0.5).astype(int)
lon5_i = np.floor(lon5 + 0.5).astype(int)
R5 = np.where((lat5_i >= LAT0) & (lat5_i <= LAT1))[0]
C5 = np.where((lon5_i >= LON0) & (lon5_i <= LON1))[0]
row_map = {r: i for i, r in enumerate(R5)}
col_map = {c: j for j, c in enumerate(C5)}
CELL5 = (lat5_i[R5] - LAT0)[:, None] * NLON + (lon5_i[C5] - LON0)[None, :]   # (len(R5), len(C5)) -> flat NCELL index

CAL_CLASSES = {"Wheat": (1, 27), "Maize": (2, 28), "Potato": (10, 36)}   # (irrigated code, rainfed code)
cols = ["cell", "row", "col", "lat", "lon", "crop", "subcrop", "area", "start", "end"]
cal = pd.read_csv(CAL_FILE, sep="\t", compression="gzip", usecols=range(10), header=0, names=cols)
allc = [c for v in CAL_CLASSES.values() for c in v]
cal = cal[cal.crop.isin(allc)].copy()
cal["i"] = (cal.row - 1).map(row_map)
cal["j"] = (cal.col - 1).map(col_map)
cal = cal.dropna(subset=["i", "j"])
cal["i"] = cal.i.astype(int); cal["j"] = cal.j.astype(int)
cal["irr"] = np.where(cal.crop <= 26, "ir", "rf")
name_of = {c: k for k, v in CAL_CLASSES.items() for c in v}
cal["crop_name"] = cal.crop.map(name_of)
print("calendar sub-crop rows in window:", len(cal))


def mirca2000_presence_1deg_no_threshold(crop_name, irr, season=None):
    g = cal[(cal.crop_name == crop_name) & (cal.irr == irr)]
    if season == 'winter':
        g = g[g.start.isin([9, 10, 11, 12])]
    elif season == 'spring':
        g = g[g.start.isin([2, 3, 4, 5])]
    cell_idx = CELL5[g.i.values, g.j.values]
    area_cell = np.bincount(cell_idx, weights=g.area.values, minlength=NCELL).reshape(NLAT, NLON)
    pres = (area_cell > 0).astype(float)
    return np.where(land_mask, pres, np.nan), np.where(land_mask, area_cell, np.nan)


presence_by_label = {}
for crop_label, (crop_name, irr, season, _old) in CROP_MAP.items():
    pres, area_cell = mirca2000_presence_1deg_no_threshold(crop_name, irr, season)
    presence_by_label[crop_label] = pres
    n_pres = int(np.nansum(pres == 1)); n_land = int(np.sum(land_mask))
    print(f"{crop_label:18s}: presence (no threshold) = {n_pres} / {n_land} land cells ({100*n_pres/n_land:.1f}%)")


calendar sub-crop rows in window: 591065
Maize_rf          : presence (no threshold) = 979 / 1332 land cells (73.5%)
Wheat-spring_rf   : presence (no threshold) = 662 / 1332 land cells (49.7%)
Wheat-winter_rf   : presence (no threshold) = 1045 / 1332 land cells (78.5%)


Potato_rf         : presence (no threshold) = 1074 / 1332 land cells (80.6%)


## 3. Save -- reuse MIRCA2000's own sowing/maturity DOY (unchanged), swap in the no-threshold presence array

In [4]:
for crop_label, (crop_name, irr, season, old_file) in CROP_MAP.items():
    old_ds = xr.open_dataset(ERA5_1DEG / old_file)
    sowing_doy = old_ds['sowing_doy'].values
    maturity_doy = old_ds['maturity_doy'].values
    lats_ref = old_ds.lat.values; lons_ref = old_ds.lon.values
    old_ds.close()

    new_presence = presence_by_label[crop_label]
    out = xr.Dataset({
        'presence': xr.DataArray(new_presence, dims=['lat', 'lon'], coords={'lat': lats_ref, 'lon': lons_ref},
                                 attrs={'long_name': f'MIRCA2000 presence, no threshold (area>0), crop={crop_label}'}),
        'sowing_doy': xr.DataArray(sowing_doy, dims=['lat', 'lon'], coords={'lat': lats_ref, 'lon': lons_ref},
                                   attrs={'long_name': 'Sowing DOY (unchanged, from MIRCA2000 5%-threshold calendar)'}),
        'maturity_doy': xr.DataArray(maturity_doy, dims=['lat', 'lon'], coords={'lat': lats_ref, 'lon': lons_ref},
                                     attrs={'long_name': 'Maturity DOY (unchanged, from MIRCA2000 5%-threshold calendar)'}),
    })
    out.attrs['description'] = (f'Presence from MIRCA2000 (no threshold) + MIRCA2000 calendar (unchanged) -- {crop_label}')
    out_path = ERA5_1DEG / f'mirca2000_no_threshold_1deg_{crop_label}.nc'
    out.to_netcdf(out_path)
    print(f"Saved: {out_path.name}  (presence cells: {int(np.nansum(new_presence==1))})")


Saved: mirca2000_no_threshold_1deg_Maize_rf.nc  (presence cells: 979)


Saved: mirca2000_no_threshold_1deg_Wheat-spring_rf.nc  (presence cells: 662)


Saved: mirca2000_no_threshold_1deg_Wheat-winter_rf.nc  (presence cells: 1045)


Saved: mirca2000_no_threshold_1deg_Potato_rf.nc  (presence cells: 1074)


## 4. Spot-check: NE-Europe/Ukraine wedge (lat 55-58, lon 30-44), Wheat-winter

In [5]:
ds = xr.open_dataset(ERA5_1DEG / 'mirca2000_no_threshold_1deg_Wheat-winter_rf.nc')
sub = ds['presence'].sel(lat=slice(55,58), lon=slice(30,44))
print("Wheat-winter presence, no-threshold MIRCA2000, NE wedge (lat 55-58, lon 30-44):")
print(sub.values)
print(f"Presence cells in wedge: {int(np.nansum(sub.values==1))} / {int(np.sum(~np.isnan(sub.values)))} land cells")
ds.close()


Wheat-winter presence, no-threshold MIRCA2000, NE wedge (lat 55-58, lon 30-44):
[[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]]
Presence cells in wedge: 60 / 60 land cells
